# 05. OpenAI 호환 서빙 — 같은 코드로 로컬 모델과 OpenAI를

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/05_openai_compatible_serving.ipynb)

> 🧭 **스토리** — 팀장의 마지막 주문. "로컬 모델이랑 OpenAI를 **같은 코드**로 바꿔 끼울 수 있게 해 줘. 다음 달엔 네이버 것도 붙일 거야." Ollama로 로컬 모델을 OpenAI 방식 API로 띄우고, `base_url` 하나만 바꿔 두 모델을 나란히 시험한다.
>
> 📖 **이론 연결** — 이론1 p103 (쓰는 방법 3가지), p56 (환각), p131~132 · p151~158 (메시지 역할·구조화 출력·소형 모델의 한계) · ⏱ **13분** · 🖥 **GPU 권장**
>
> GPU(T4)가 있으면 빠릅니다. 없어도 CPU로 돌아갑니다(느림).
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
1. `openai` 패키지와 실습 데이터 로더
2. **Ollama** 설치·서버 실행(코랩 리눅스 VM 안에서) → EXAONE 4.0 1.2B의 **GGUF Q4_K_M(812MB)** 받기. 노트북 04 7절의 그 파일입니다.
3. OpenAI 키 입력, 백엔드 표(`BACKENDS`)와 비용 계산 함수

Ollama는 GPU가 있으면 GPU를 쓰고, 없으면 CPU로도 1.2B 4비트 모델을 돌립니다.

In [ ]:
%pip -q install openai

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

In [ ]:
# ⚙️ 실습 데이터 로더 — 깃허브(gunyang-ai-api/data)에서 자동으로 받아옵니다. 업로드 불필요.
import os, json, urllib.request
from pathlib import Path
DATA_BASE = os.environ.get("GENAI_BASE", "https://raw.githubusercontent.com/acho98/gunyang-ai-api/main/data/")
CACHE = Path("genai_data"); CACHE.mkdir(exist_ok=True)

def fetch(name: str) -> str:
    """파일 이름(예: 'hanbit_rules.pdf', 'scan/p2.png') → 로컬 경로. URL이면 내려받아 캐시."""
    if DATA_BASE.startswith("http"):
        dst = CACHE / name
        if not dst.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(DATA_BASE + name, dst)
        return str(dst)
    return str(Path(DATA_BASE) / name)

def load_json(name: str):
    with open(fetch(name), encoding="utf-8") as f:
        return json.load(f)

def load_text(name: str) -> str:
    with open(fetch(name), encoding="utf-8") as f:
        return f.read()

# 한글 폰트(차트용) — gunyang-data의 BMDOHYEON.ttf 를 자동으로 받아 적용. 실패해도 진행.
def set_korean_font():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager as fm
    font_url = "https://raw.githubusercontent.com/acho98/gunyang-data/main/BMDOHYEON.ttf"
    dst = CACHE / "BMDOHYEON.ttf"
    try:
        if not dst.exists():
            local = Path(DATA_BASE).parent / "BMDOHYEON.ttf" if not DATA_BASE.startswith("http") else None
            if local and local.exists():
                dst.write_bytes(local.read_bytes())
            else:
                urllib.request.urlretrieve(font_url, dst)
        fm.fontManager.addfont(str(dst))
        plt.rcParams["font.family"] = fm.FontProperties(fname=str(dst)).get_name()
        plt.rcParams["axes.unicode_minus"] = False
    except Exception as e:
        print("(안내) 한글 폰트 설정 실패 — 영문 라벨은 정상:", e)
set_korean_font()
print("데이터 소스:", DATA_BASE)

In [ ]:
# ⚙️ Ollama 설치 + 서버 실행 (코랩 리눅스 VM 안에서). 첫 실행 1분 안팎.
import subprocess, shutil, time, os, urllib.request
if not shutil.which("ollama"):
    if not shutil.which("zstd"):   # 설치 파일이 .tar.zst 형식이라 압축 해제 도구 zstd가 필요합니다(코랩 기본 이미지에 없음)
        subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-800:])
        raise RuntimeError("Ollama 설치 실패: 위 설치 로그를 강사에게 보여 주세요.")
# 서버를 백그라운드로 띄우고 준비될 때까지 기다립니다
if not os.environ.get("_OLLAMA_STARTED"):
    # 모델을 1시간 동안 GPU에 둔다(기본 5분이 지나면 내려가고, 다시 올리는 데 코랩 T4에서 1분 넘게 걸림)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     env=dict(os.environ, OLLAMA_KEEP_ALIVE="1h"))
    os.environ["_OLLAMA_STARTED"] = "1"
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("Ollama 준비 완료:", subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ⚙️ 모델 받기 — GGUF(Q4_K_M 양자화) 를 허깅페이스에서 Ollama로 바로 당겨옵니다
MODEL_LOCAL = "hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M"
import subprocess, time
t0 = time.time()
r = subprocess.run(["ollama", "pull", MODEL_LOCAL], capture_output=True, text=True)
print(r.stdout[-300:] if r.returncode == 0 else r.stderr[-500:])
print(f"pull 소요 {time.time()-t0:.0f}초")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
# 첫 요청은 모델을 GPU에 올리고 계산 커널을 준비하느라 1분 안팎 걸립니다(코랩 T4 실측).
# 여기서 긴 프롬프트로 토큰 1개를 미리 생성해 두면 이후 질문은 1초 안팎에 답합니다.
import json, urllib.request
t1 = time.time()
urllib.request.urlopen(urllib.request.Request("http://localhost:11434/api/generate",
    data=json.dumps({"model": MODEL_LOCAL, "prompt": "한빛테크 규정 " * 300, "stream": False,
                     "options": {"num_predict": 1}}).encode()), timeout=900).read()
print(f"GPU 적재·준비 {time.time()-t1:.0f}초 (이후 요청은 바로 답합니다)")

In [ ]:
# ⚙️ OpenAI API 키 입력 — 수업에서 나눠 준 키를 붙여 넣으세요. (화면에 표시되지 않습니다)
# 규칙: 키를 코드에 직접 적지 말 것 · 출력하지 말 것 · 수업 후 키는 폐기됩니다.
# 입력한 키는 모델 조회 API(GET /v1/models/{모델} = client.models.retrieve)로 바로 확인합니다. 토큰을 쓰지 않아 비용이 없습니다.
import os, json, urllib.request, urllib.error
from getpass import getpass
MODEL_API = "gpt-6-luna"        # 수업용 소형 모델 ($0.10 / $0.50 per 1M tokens). 바뀌면 여기만 수정.

def check_key(key: str):
    """키 확인 → (상태, 안내). 키 문자열과 오류 본문은 출력하지 않는다(유형·상태 코드만)."""
    req = urllib.request.Request(f"https://api.openai.com/v1/models/{MODEL_API}",
                                 headers={"Authorization": f"Bearer {key}"})
    try:
        with urllib.request.urlopen(req, timeout=15):
            return "ok", "키 확인 완료"
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode()).get("error", {})        # 판단에만 쓰고 출력하지 않는다
        except Exception:
            err = {}
        code, msg = err.get("code"), str(err.get("message", "")).lower()
        scope = "insufficient permissions" in msg or "missing scopes" in msg   # 권한 범위가 좁은 제한 키
        if e.code == 401 and not scope and code in ("invalid_api_key", None):
            return "bad", "키가 올바르지 않습니다(401). 빠진 글자·앞뒤 공백 없이 다시 붙여 넣으세요."
        if e.code in (401, 403):
            return "ok", f"키는 유효하지만 모델 조회 권한이 없습니다({e.code}). 실습은 진행하고, 호출이 막히면 강사에게 알려 주세요."
        if e.code == 404:
            return "ok", f"키는 유효하지만 모델 {MODEL_API}에 접근할 수 없습니다(404). 강사에게 알려 주세요."
        if e.code == 429:
            return "ok", "키는 유효합니다. 요청 한도에 걸렸습니다(429). 잠시 뒤 다시 실행하세요."
        return "err", f"확인 실패: HTTPError {e.code}. 잠시 뒤 이 셀을 다시 실행하세요."
    except Exception as e:
        return "err", f"확인 실패: {type(e).__name__}. 네트워크 연결을 확인하고 이 셀을 다시 실행하세요."

key = os.environ.get("OPENAI_API_KEY", "").strip()
for attempt in range(1, 4):
    if not key:
        key = getpass(f"OpenAI API 키 ({attempt}/3): ").strip()
    status, msg = check_key(key)
    print(msg)
    if status != "bad":
        break
    key = ""                                   # 틀린 키는 버리고 다시 입력받는다
if status == "bad":
    os.environ.pop("OPENAI_API_KEY", None)
    raise RuntimeError("키 확인 3회 실패: 강사에게 키를 다시 받은 뒤 이 셀을 다시 실행하세요.")
os.environ["OPENAI_API_KEY"] = key
print("모델:", MODEL_API)

In [ ]:
# ⚙️ 같은 클라이언트, 다른 백엔드 — base_url 만 바꾸면 로컬 모델 / OpenAI / (3차시) 네이버 CLOVA Studio 를 같은 코드로 부릅니다
from openai import OpenAI
BACKENDS = {
    "openai": dict(base_url=None, api_key=os.environ.get("OPENAI_API_KEY", "none"), model="gpt-6-luna"),
    "local":  dict(base_url="http://localhost:11434/v1", api_key="ollama", model="hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M"),
    # 3차시 예고: 네이버 CLOVA Studio 는 OpenAI 호환 엔드포인트를 제공합니다 (키는 3차시에 발급)
    "clova":  dict(base_url="https://clovastudio.stream.ntruss.com/v1/openai", api_key=os.environ.get("CLOVA_API_KEY", "none"), model="HCX-005"),
}
def make_client(backend: str):
    cfg = BACKENDS[backend]
    client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"]) if cfg["base_url"] else OpenAI(api_key=cfg["api_key"])
    return client, cfg["model"]

In [ ]:
# ⚙️ 비용 계산 — 2026-10 기준 공식 단가(USD per 1M tokens). https://developers.openai.com/api/docs/pricing
PRICES = {  # (입력, 캐시 입력, 출력)
    "gpt-6-luna":   (0.10, 0.01, 0.50),
    "gpt-5-nano":   (0.05, 0.005, 0.40),
    "gpt-5.6-luna": (0.20, 0.02, 1.20),
    "gpt-6-astra":  (10.0, 1.0, 50.0),
    "text-embedding-3-small": (0.02, 0.02, 0.0),
}
CACHE_WRITE_X = {"gpt-6-luna": 1.25, "gpt-5.6-luna": 1.25, "gpt-6-astra": 1.25}   # GPT-5.6 이후 모델: 캐시 쓰기 = 입력 단가 × 1.25
def cost_usd(model: str, in_tok: int, out_tok: int = 0, cached_tok: int = 0, write_tok: int = 0) -> float:
    """in_tok = usage.input_tokens(캐시 읽기 cached_tok · 캐시 쓰기 write_tok 포함 전체 입력)."""
    p_in, p_cache, p_out = PRICES.get(model, (0.10, 0.01, 0.50))
    p_write = p_in * CACHE_WRITE_X.get(model, 1.0)
    return ((in_tok - cached_tok - write_tok) * p_in + write_tok * p_write + cached_tok * p_cache + out_tok * p_out) / 1_000_000
def cost_krw(usd: float, rate: float = 1400.0) -> float:
    return usd * rate

## 1. 같은 클라이언트, `base_url`만 다르게 🔵
Ollama는 `http://localhost:11434/v1`에 **OpenAI와 같은 모양의 API**를 엽니다(`/v1/chat/completions`, `/v1/responses`, `/v1/embeddings`, `/v1/models`). 키는 아무 문자열이나 됩니다.
먼저 `/v1/models`로 로컬 서버에 어떤 모델이 있는지 OpenAI 클라이언트로 물어봅니다.

In [ ]:
local_client, local_model = make_client("local")
print("로컬 서버 모델 목록:", [m.id for m in local_client.models.list().data])
print("로컬 백엔드 모델:", local_model)

## 2. 그냥 부르면? — 사고 모드가 새어 나온다 🔵
아무 옵션 없이 로컬 EXAONE을 불러 봅니다.

In [ ]:
import time
msgs = [{"role": "user", "content": "한빛테크 신입 사원에게 출장비 정산 요령을 한 문장으로 알려줘."}]
for label, extra in [("옵션 없음(기본)", {}), ('reasoning_effort="none"', {"reasoning_effort": "none"})]:
    t0 = time.time()
    r = local_client.chat.completions.create(model=local_model, messages=msgs, max_tokens=600, **extra)
    print(f"=== {label}: {time.time()-t0:.1f}초 · 출력 {r.usage.completion_tokens}토큰 ===")
    print(r.choices[0].message.content, "\n")

💡 **해석** — 답 앞에 "Okay, 사용자가 … 요청했네" 같은 **혼잣말**이 붙고 `</think>` 뒤에야 진짜 답이 나옵니다. 이 GGUF는 **사고 모드가 기본으로 켜져** 있고(노트북 03 6절), Ollama가 사고 부분을 따로 떼지 않고 본문에 넣어 돌려주기 때문입니다. 강사 환경에서 기본 호출의 출력은 **66~245토큰**으로 실행마다 크게 달랐고, 길 때는 대부분이 사고였습니다.
OpenAI API와 같은 이름의 옵션 **`reasoning_effort="none"`**을 주면 사고 없이 답만 나옵니다. 강사 환경에서는 **20~40토큰 · 0.5초 안팎**이었습니다. 코랩 T4(2026-10-17)에서는 이 첫 호출만 13초가 걸렸고(사고 끔 경로의 첫 계산 준비), 이후 같은 방식의 호출은 0.2초였습니다. **Ollama는 OpenAI 호환 `reasoning_effort`를 사고 모드 on/off로 매핑합니다(0.35 · 0.40 실측).** 아래 `chat()` 헬퍼는 이 옵션을 두 백엔드 모두에 기본으로 넣고, 혹시 남은 `</think>` 앞부분도 잘라 냅니다. 로컬 호출에는 출력 상한 `max_tokens=600`도 둡니다.

## 3. `chat()` 헬퍼 — 백엔드 이름만 바꿔 부르기 🔵
- `reasoning_effort="none"`: GPT-6 계열은 이 값이 `"none"`일 때만 `temperature`를 받습니다. 로컬 EXAONE에는 "사고 끄기"로 작동합니다.
- API 호출은 실패할 수 있습니다(키 오류·네트워크). 실패하면 이유를 한 번 출력하고, 그 백엔드는 이후 셀에서 건너뜁니다.
- `FOLD_SYSTEM`은 4절에서 쓰는 장치입니다. 지금은 비어 있습니다.

In [ ]:
CLIENTS = {b: make_client(b) for b in ["local", "openai"]}
FAILED = {}                                     # 실패한 백엔드 → 이유 (같은 오류를 반복해서 찍지 않으려고)
FOLD_SYSTEM = set()                             # system 지시를 user 메시지 안으로 옮겨 보낼 백엔드 (4절에서 채웁니다)

def chat(backend, messages, **kw):
    """(답, usage, 초) 반환. 실패하면 (None, None, 0)."""
    if backend in FAILED:
        return None, None, 0.0
    client, model = CLIENTS[backend]
    if backend in FOLD_SYSTEM and len(messages) > 1 and messages[0]["role"] == "system":
        first = messages[1]
        messages = [dict(first, content=f"[지시]\n{messages[0]['content']}\n\n[질문]\n{first['content']}")] + messages[2:]
    kw.setdefault("reasoning_effort", "none")
    if backend != "openai":
        kw.setdefault("max_tokens", 600)        # 로컬 모델이 길게 늘어놓지 않도록 출력 상한
    t0 = time.time()
    try:
        r = client.chat.completions.create(model=model, messages=messages, **kw)
    except Exception as e:
        FAILED[backend] = f"{type(e).__name__} {getattr(e, 'status_code', '')}".strip()   # 메시지 본문은 찍지 않음(키 일부가 보일 수 있음)
        print(f"[{backend}] 호출 실패 — 키·네트워크를 확인하세요. 이 백엔드는 이후 건너뜁니다. ({FAILED[backend]})")
        return None, None, 0.0
    text = (r.choices[0].message.content or "").split("</think>")[-1].strip()
    return text, r.usage, time.time() - t0

def show(backend, text, usage, sec):
    if text is None:
        print(f"[{backend}] (건너뜀)"); return
    cost = f" · {cost_usd(MODEL_API, usage.prompt_tokens, usage.completion_tokens):.6f} USD" if backend == "openai" else " · 0원(로컬)"
    print(f"[{backend}] {sec:.1f}초 · 입력 {usage.prompt_tokens} / 출력 {usage.completion_tokens}토큰{cost}\n  {text}\n")

for b in ["local", "openai"]:
    show(b, *chat(b, msgs))

💡 **해석** — **같은 `messages`, 같은 함수**로 두 모델을 불렀습니다. 바뀐 것은 `BACKENDS` 표의 `base_url`과 모델 이름뿐입니다. 이것이 "OpenAI 호환"의 뜻이고, 3차시에 네이버 CLOVA Studio를 붙일 때도 이 표에 한 줄을 더하면 됩니다. 지연 시간과 토큰 수를 비교해 보세요. 강사 노트북에서 로컬 EXAONE은 **0.2~0.5초 · 출력 20~40토큰**이었습니다. 로컬은 네트워크 왕복이 없고 **비용이 0원**입니다. API는 모델이 크고 한국어 답이 더 정돈된 경우가 많습니다.

## 4. 실험 ① 시스템 지시가 있을 때와 없을 때 🔵
`system` 역할 메시지는 "어떤 말투·형식으로 답할지"를 정합니다(이론1 p131~132). 답과 함께 **입력 토큰 수**도 찍습니다. 지시가 모델에 실제로 들어갔다면 입력 토큰이 늘어야 합니다.

In [ ]:
SYSTEM = "너는 한빛테크 인사팀 도우미다. 존댓말로 두 문장 이내로 답하고, 끝에 '— 인사팀'이라고 서명한다."
user = {"role": "user", "content": "해외 출장 갈 때 여행자보험은 누가 내나요?"}

def system_test(b):
    t0, u0, _ = chat(b, [user], temperature=0)
    t1, u1, _ = chat(b, [{"role": "system", "content": SYSTEM}, user], temperature=0)
    if t0 is None or t1 is None:
        print(f"[{b}] (건너뜀)"); return
    print(f"[{b} · 지시 없음] 입력 {u0.prompt_tokens}토큰 → {t0[:150]}")
    print(f"[{b} · 지시 있음] 입력 {u1.prompt_tokens}토큰 → {t1[:150]}")
    print("   ⚠️ 입력 토큰이 같다 = 시스템 지시가 모델에 전달되지 않았다!\n" if u0.prompt_tokens == u1.prompt_tokens
          else "   ✅ 지시가 전달되었다(입력 토큰 증가)\n")

for b in ["local", "openai"]:
    system_test(b)

💡 **해석** — 강사 환경(Ollama 0.35.0)에서는 로컬 EXAONE의 입력 토큰이 **지시가 있든 없든 같았습니다**. Ollama가 이 GGUF의 채팅 템플릿을 처리하면서 **첫 system 메시지를 빠뜨린** 것입니다. 오류도 경고도 없이 조용히 무시됩니다. OpenAI는 지시 길이만큼 입력 토큰이 늘어납니다.
"OpenAI 호환"은 **요청 모양이 같다**는 뜻이지 **동작이 같다**는 보장이 아닙니다. 노트북 03에서 본 것처럼 결국 모델이 받는 것은 템플릿이 만든 문자열 하나입니다. `usage.prompt_tokens`로 확인하는 습관을 들이세요. (Ollama 버전이 올라가 고쳐졌다면 ✅가 보일 수 있습니다.)

**우회법**: 지시를 user 메시지 앞에 붙여 보냅니다. `FOLD_SYSTEM`에 `"local"`을 넣으면 `chat()`이 자동으로 그렇게 합니다. 이후 실험은 이 상태로 진행합니다.

In [ ]:
FOLD_SYSTEM.add("local")          # 로컬 백엔드는 system 지시를 user 메시지 안에 넣어 보냅니다
system_test("local")

💡 **해석** — 이제 입력 토큰이 늘고 답의 말투와 길이가 바뀝니다. 강사 환경에서는 23 → **66토큰**으로 늘었고, 답이 "… 한빛테크 인사팀에서 처리해 드립니다. — 인사팀" 한 문장으로 바뀌었습니다. 그래도 작은 로컬 모델은 "두 문장 이내", "서명" 같은 **형식 지시를 일부만** 지키는 경우가 많습니다. 같은 지시라도 **모델 크기에 따라 지키는 정도가 다르다**는 것이 이론1 p154~158 "소형 모델의 한계"입니다.

## 5. 실험 ② "JSON만 출력해" — 5번 중 몇 번 지킬까 🔵
프로그램이 모델 답을 받아 쓰려면 **형식이 매번 맞아야** 합니다. 같은 지시를 백엔드마다 5번씩(`temperature=0.7`) 보내 `json.loads`가 그대로 성공하는 비율을 셉니다.

In [ ]:
import json
JSON_SYS = ('다음 문장에서 직급과 출장 지역을 뽑아 JSON 하나만 출력하라. 설명·코드블록 없이 '
            '{"grade": "...", "region": "..."} 형식만 쓴다.')
JSON_USER = "다음 주에 김 과장이 유럽 지사로 출장을 갑니다."
N_TRY = 5

def json_rate(backend):
    ok, sample_bad = 0, None
    for _ in range(N_TRY):
        text, usage, sec = chat(backend, [{"role": "system", "content": JSON_SYS},
                                          {"role": "user", "content": JSON_USER}], temperature=0.7)
        if text is None:
            return None, None
        try:
            json.loads(text); ok += 1
        except json.JSONDecodeError:
            sample_bad = sample_bad or text
    return ok, sample_bad

for b in ["local", "openai"]:
    ok, bad = json_rate(b)
    if ok is None:
        print(f"[{b}] (건너뜀)"); continue
    print(f"[{b}] 파싱 성공 {ok}/{N_TRY}" + (f"\n  실패 예: {bad[:150]!r}" if bad else ""))

💡 **해석** — "JSON만 출력하라"는 **부탁**일 뿐이라 모델이 어길 수 있습니다. 강사 환경 로컬 EXAONE은 두 번 돌려 **5/5, 4/5**였고, 실패 1회는 답을 ` ```json … ``` ` 코드블록으로 감싼 경우였습니다. 내용은 맞아도 형식이 한 글자만 어긋나면 `json.loads`는 실패합니다. (4절의 우회법 전에는 지시가 아예 전달되지 않아 **0/5**였습니다.) 5번은 표본이 작아 매번 결과가 달라질 수 있습니다. 실무에서는 부탁 대신 **구조화 출력**(JSON 스키마를 강제하는 API 기능, 이론1 p151·노트북 12)을 쓰고, 그래도 파싱 실패에 대비한 재시도 코드를 둡니다.

## 6. 실험 ③ 환각 — 모르는 것을 물으면 🔵
규정집에는 회사 매출이 없습니다. 근거 없이 물으면 어떻게 답할까요? 그리고 규정집 **제2장(국내 출장)**을 붙여 넣고 같은 질문 + 규정집에 있는 질문을 하면 어떻게 바뀔까요?

In [ ]:
rules = load_text("hanbit_rules_gt.md")
ch2 = rules[rules.index("## 제2장"):rules.index("## 제3장")]          # 제2장 국내 출장 (조문 + 표 1)
print(f"붙여 넣을 근거: 제2장 {len(ch2)}자\n{ch2[:200]}…\n")

QS = ["한빛테크의 2025년 매출은 얼마인가요?", "한빛테크 과장의 국내 출장 숙박비 상한은 1박에 얼마인가요?"]
GROUND = "아래 [규정]에 있는 내용으로만 한 문장으로 답하라. 규정에 없으면 '규정집에 없는 내용입니다'라고 답하라.\n\n[규정]\n"
for b in ["local", "openai"]:
    for q in QS:
        bare, _, _ = chat(b, [{"role": "user", "content": q}], temperature=0)
        grounded, _, _ = chat(b, [{"role": "system", "content": GROUND + ch2}, {"role": "user", "content": q}], temperature=0)
        if bare is None:
            continue
        print(f"[{b}] Q. {q}\n  근거 없이 → {bare[:160]}\n  제2장 붙임 → {grounded[:160]}\n")

💡 **해석** — 강사 환경 로컬 EXAONE의 결과입니다.

| 질문 | 근거 없이 | 제2장 붙임 |
|---|---|---|
| 2025년 매출 | "공식 발표를 통해 확인할 수 있습니다…" (일반론으로 둘러댐) | "규정집에 … 매출에 대한 내용이 없습니다" |
| 과장 숙박비 상한 | "**1박당 10만 원**입니다" (틀림, 자신 있게) | "1박 **110,000원**" (표 1과 일치) |

근거가 없으면 모델은 **그럴듯한 숫자를 자신 있게 지어냅니다**(이론1 p56 "환각은 구조"). 근거를 붙이고 "없으면 없다고 하라"고 지시하면 맞는 값을 찾거나 모른다고 답합니다. 문서 전체를 매번 붙이면 비용과 컨텍스트가 커지므로(노트북 02), **질문에 맞는 조각만 찾아 붙이는 것**이 RAG입니다. 노트북 11에서 만듭니다.

## 7. `/v1/embeddings` — 로컬 서버에 임베딩도 있을까 🔵
Ollama의 OpenAI 호환 API에는 임베딩 엔드포인트도 있습니다. 같은 EXAONE 모델로 불러 봅니다.

In [ ]:
try:
    e = local_client.embeddings.create(model=local_model, input=["출장비 정산", "숙박비 상한"])
    print(f"임베딩 {len(e.data)}개 · 차원 {len(e.data[0].embedding)}")
except Exception as ex:
    print("임베딩 실패:", type(ex).__name__, str(ex)[:200])
    print("→ 이 GGUF는 생성용 모델이라 임베딩을 지원하지 않습니다. 임베딩은 전용 모델로 만듭니다(노트북 09).")

💡 **해석** — 강사 환경에서는 `501 This server does not support embeddings`가 났습니다. **엔드포인트가 있다고 모든 모델이 지원하는 것은 아닙니다.** 생성 모델과 임베딩 모델은 역할이 다르고, 임베딩은 검색 품질을 위해 따로 학습된 전용 모델(e5·KURE 등, 노트북 09)을 씁니다.

## 8. 3차시 예고 — 네이버 CLOVA Studio도 같은 코드로 🔵
`BACKENDS["clova"]`에 이미 `base_url`(`https://clovastudio.stream.ntruss.com/v1/openai`)과 모델 `HCX-005`가 들어 있습니다. 키는 **3차시에 발급**하므로 지금은 키가 없으면 아무것도 하지 않습니다.

In [ ]:
if os.environ.get("CLOVA_API_KEY"):
    clova_client, clova_model = make_client("clova")
    r = clova_client.chat.completions.create(model=clova_model, messages=msgs)
    print("[clova]", r.choices[0].message.content)
else:
    print("CLOVA_API_KEY 없음 — 3차시에 키를 받은 뒤 이 셀을 다시 실행하세요. (코드는 local·openai와 똑같습니다)")

> 📌 **장표에서 소개**: vLLM·`transformers serve`도 OpenAI 호환 서버를 띄우지만 설치가 무거워 실습에서는 Ollama만 씁니다. gpt-oss-20b(약 14GB)·qwen3:30b(약 19GB) 같은 20B급 모델은 무료 T4(15GB)에 올라가지 않습니다.

## 9. 바꿔서 실행 🟡
1. **세 번째 백엔드**: `PULL_QWEN = True`로 바꾸고 실행하면 `qwen3:4b`(2.5GB, 툴 호출 지원 모델)를 받아 시스템 지시·JSON 실험을 세 백엔드로 돌립니다. 다운로드에 1~2분 걸립니다. qwen3:4b는 `reasoning_effort="none"`을 줘도 사고를 하므로(답 앞에 `</think>`까지 생각을 씀) 더 느립니다. 시스템 지시는 그대로 전달되는지도 확인하세요.
2. **temperature**: `TEMP`를 0, 0.7, 1.5로 바꿔 같은 질문을 3번씩 보내고 답이 얼마나 흔들리는지 보세요.

In [ ]:
PULL_QWEN = False                                  # ← True 로 바꾸면 qwen3:4b(2.5GB)를 받습니다
if PULL_QWEN:
    subprocess.run(["ollama", "pull", "qwen3:4b"], capture_output=True, text=True)
    BACKENDS["qwen"] = dict(base_url="http://localhost:11434/v1", api_key="ollama", model="qwen3:4b")
    CLIENTS["qwen"] = make_client("qwen")
    system_test("qwen")
    for b in ["local", "qwen", "openai"]:
        ok, bad = json_rate(b)
        print(f"[{b}] JSON 파싱 성공 {ok}/{N_TRY}" if ok is not None else f"[{b}] (건너뜀)")
else:
    print("PULL_QWEN = False — 세 번째 백엔드를 건너뜁니다.")

In [ ]:
TEMP = 1.5                                         # ← 0, 0.7, 1.5 로 바꿔 보세요
for b in ["local", "openai"]:
    for i in range(3):
        text, usage, sec = chat(b, msgs, temperature=TEMP)
        if text is not None:
            print(f"[{b} · T={TEMP} · {i+1}] {text[:150]}")

### 🤔 생각해보기

**로컬 1.2B가 API(gpt-6-luna)를 이긴 항목, 진 항목은?** 이 노트북의 결과로 표를 채워 보세요.

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

| 항목 | 유리한 쪽 | 이유 |
|---|---|---|
| 비용 | 로컬 | 토큰 단가 0원. 대신 GPU·서버 시간이 든다(이론1 p106) |
| 데이터 반출 | 로컬 | 문서가 회사 밖으로 나가지 않는다 |
| 지연(짧은 답) | 비슷하거나 로컬 | 네트워크 왕복이 없다. 단 GPU가 없으면 느리다 |
| 형식 준수(JSON·지시) | API | 큰 모델이 지시를 더 잘 지킨다(실험 ①②) |
| 한국어 품질·지식 | API | 파라미터와 학습 데이터 규모 차이 |
| 환각 | 둘 다 위험 | 근거를 붙이면 둘 다 나아진다(실험 ③) → RAG |
| 라이선스 | API(또는 Apache 모델) | EXAONE 1.2B는 비상업 라이선스 |

결론: **"무엇을 시킬지"에 따라 고른다.** 분류·추출 같은 단순·대량 작업은 로컬, 형식과 품질이 중요한 최종 답변은 API. 같은 코드로 바꿔 끼울 수 있으니 섞어 쓰기(라우팅)도 쉽습니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- Ollama는 로컬 모델을 OpenAI 호환 API(`/v1`)로 연다. `base_url`만 바꾸면 같은 코드로 로컬·OpenAI·CLOVA를 부른다.
- Ollama의 EXAONE GGUF는 사고 모드가 기본이다. `reasoning_effort="none"`으로 끈다.
- 시스템 지시·JSON 형식은 작은 모델일수록 덜 지킨다. 형식이 중요하면 구조화 출력과 재시도를 쓴다.
- 근거 없이 물으면 지어낸다. 근거를 붙이면 바뀐다. 필요한 조각만 찾아 붙이는 것이 RAG다.
- 엔드포인트가 있어도 모델이 지원해야 한다. 임베딩은 전용 모델로.

> 다음 노트북: **06_pdf_parsers_ocr — PDF 파서 3종과 기본 OCR**